In [1]:
!pip install -q transformers datasets peft accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.5 MB/s eta 0:00:00:00:0100:01


In [2]:
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
from peft import LoraConfig, get_peft_model, TaskType
import numpy as np
from sklearn.metrics import f1_score

In [3]:
from huggingface_hub import login

login("")

In [4]:
dataset = load_dataset("uitnlp/vigoemotions")

README.md: 0.00B [00:00, ?B/s]

train.csv: 0.00B [00:00, ?B/s]

val.csv: 0.00B [00:00, ?B/s]

test.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/16531 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2066 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2067 [00:00<?, ? examples/s]

In [5]:
import re

NUM_LABELS = 28  # ViGoEmotions

def parse_labels(label_str):
    indices = list(map(int, re.findall(r"\d+", label_str)))
    multi_hot = [0.0] * NUM_LABELS
    for idx in indices:
        multi_hot[idx] = 1.0
    return multi_hot

def preprocess(example):
    example["labels"] = parse_labels(example["labels"])
    return example

dataset = dataset.map(preprocess)

Map:   0%|          | 0/16531 [00:00<?, ? examples/s]

Map:   0%|          | 0/2066 [00:00<?, ? examples/s]

Map:   0%|          | 0/2067 [00:00<?, ? examples/s]

In [6]:
# Tiền xử lý dữ liệu text
# 1. Tự xây dựng từ điển chuẩn hóa Teencode (Manually curated dictionary)
teencode_dict = {
    # 1. Nhóm phủ định và đồng ý
    "ko": "không", "k": "không", "khong": "không", "khg": "không", "hong": "không", "hông": "không",
    "chả": "chẳng", "uk": "ừ", "ukm": "ừ", "uh": "ừ",
    "đéo": "không", "đ": "đéo", # Nhóm từ mang sắc thái tiêu cực mạnh xuất hiện trong data

    # 2. Nhóm đại từ nhân xưng (Gây lỗi phân loại nhiều nhất theo Bảng 13)
    "t": "tao", "tui": "tôi", "mik": "mình", "m": "mày", "th": "thằng",
    "c": "chị", "e": "em", "ae": "anh em", "ce": "chị em", "mn": "mọi người",
    "vk": "vợ", "ck": "chồng", "hs": "học sinh", "bs": "bác sĩ",

    # 3. Nhóm tính từ / phó từ / cảm thán thường dùng (Bảng 10 & 13)
    "z": "vậy", "zậy": "vậy", "v": "vậy", "dzậy": "vậy",
    "vs": "với", "cx": "cũng", "r": "rồi", "rùi": "rồi", "ròi": "rồi",
    "nhìu": "nhiều", "ms": "mới", "mí": "mới", "lun": "luôn", "luân": "luôn",
    "quá": "quá", "qá": "quá", "ntn": "như thế nào", "bh": "bao giờ", "bây h": "bây giờ",
    "vãi": "rất", "vl": "rất", "vcl": "rất", "đỉnh": "tuyệt", "haha": "cười", "hihi": "cười",

    # 4. Nhóm động từ và danh từ phổ biến
    "dc": "được", "đc": "được", "dk": "được", "đx": "đã", "đung": "đúng",
    "thik": "thích", "pt": "biết", "bt": "biết", "ns": "nói", "lm": "làm",
    "nt": "nhắn tin", "đg": "đang", "rep": "trả lời", "vn": "việt nam",

    # 5. Nhóm từ để hỏi
    "j": "gì", "s": "sao", "ntn": "như thế nào", "chi": "gì"
}



def normalize_social_media_text(text):
    # a. Thu gọn ký tự chữ cái kéo dài (vd: cườiiiii -> cười, đẹpppp -> đẹp)
    # Giữ lại tối đa 1 ký tự (tiếng Việt không có từ nào có 2 chữ cái giống hệt nhau đi liền trừ một số từ mượn/đặc biệt)
    text = re.sub(r'([A-Za-zàáảãạâấầẩẫậăắằẳẵặèéẻẽẹêếềểễệìíỉĩịòóỏõọôốồổỗộơớờởỡợùúủũụưứừửữựỳýỷỹỵđÀÁẢÃẠÂẤẦẨẪẬĂẮẰẲẴẶÈÉẺẼẸÊẾỀỂỄỆÌÍỈĨỊÒÓỎÕỌÔỐỒỔỖỘƠỚỜỞỠỢÙÚỦŨỤƯỨỪỬỮỰỲÝỶỸỴĐ])\1{2,}', r'\1', text)

    # b. Thu gọn dấu câu kéo dài (vd: :))))) -> :)) theo đúng mô tả của bài báo
    text = re.sub(r'(\)|\(|!|\?|\.){3,}', r'\1\1', text)

    # c. Chuẩn hóa teencode bằng từ điển
    words = text.split()
    normalized_words = []
    for w in words:
        # Tách các dấu câu dính liền với chữ để tra từ điển cho chuẩn
        clean_word = re.sub(r'[^\w\s]', '', w).lower()
        if clean_word in teencode_dict:
            # Thay thế bằng từ chuẩn nhưng cố gắng giữ lại dấu câu (nếu có)
            normalized_words.append(w.lower().replace(clean_word, teencode_dict[clean_word]))
        else:
            normalized_words.append(w)

    # Nối lại thành câu (Lưu ý: EMOJI VẪN ĐƯỢC GIỮ NGUYÊN HOÀN TOÀN)
    return " ".join(normalized_words)

In [7]:
def apply_text_preprocess(example):
    example['text'] = normalize_social_media_text(example['text'])
    return example

In [8]:
dataset = dataset.map(apply_text_preprocess)

Map:   0%|          | 0/16531 [00:00<?, ? examples/s]

Map:   0%|          | 0/2066 [00:00<?, ? examples/s]

Map:   0%|          | 0/2067 [00:00<?, ? examples/s]

In [9]:
print(dataset["train"])

Dataset({
    features: ['id', 'text', 'labels'],
    num_rows: 16531
})


In [10]:
import numpy as np

def count_samples_per_class(dataset):
    # Example: each row is a sample, each column is a class
    labels = np.array(dataset["labels"])

    # Count number of samples per class
    class_counts = labels.sum(axis=0)
    return class_counts

print(count_samples_per_class(dataset["train"]))

[2868.  824. 1614. 1175.  762. 1046. 1062.  774.  883.  765.  635.  917.
  687.  651.  840.  676.  745.  728.  707.  885. 1522. 2785.  747. 1206.
 1589. 2662.  969.  821.]


In [11]:
def compute_weight_for_classes_in_training(dataset):
    samples_count_per_class = count_samples_per_class(dataset)
    total_samples = len(dataset)
    weight_per_class = np.log((total_samples-samples_count_per_class)/samples_count_per_class)
    return weight_per_class

print(compute_weight_for_classes_in_training(dataset["train"])[1])

2.9476912219335607


In [12]:
model_name = ("uitnlp/visobert")

tokenizer = AutoTokenizer.from_pretrained("uitnlp/visobert")
def tokenize(example):
   return tokenizer(
        example["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

dataset = dataset.map(tokenize, batched=True)
dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

config.json:   0%|          | 0.00/644 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/471k [00:00<?, ?B/s]

Map:   0%|          | 0/16531 [00:00<?, ? examples/s]

Map:   0%|          | 0/2066 [00:00<?, ? examples/s]

Map:   0%|          | 0/2067 [00:00<?, ? examples/s]

In [13]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=NUM_LABELS,
    problem_type="multi_label_classification"
    #id2label = {0: "negative", 1: "neutral", 2: "positive"}
)

model = model.to("cuda")

pytorch_model.bin:   0%|          | 0.00/390M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: uitnlp/visobert
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


model.safetensors:   0%|          | 0.00/390M [00:00<?, ?B/s]

In [14]:
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=16,
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=["query", "key", "value", "dense"]  # attention layers
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 3,266,332 || all params: 100,853,816 || trainable%: 3.2387


In [15]:
from transformers import Trainer
import torch.nn as nn

loss_fct = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(compute_weight_for_classes_in_training(dataset["train"]),dtype=torch.float32).to('cuda'))

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        loss = loss_fct(logits, labels)

        return (loss, outputs) if return_outputs else loss

In [16]:
def tune_thresholds(probs, labels, prev_thresholds=None, delta=0.1):
    num_classes = probs.shape[1]

    # Initialize if first epoch
    if prev_thresholds is None:
        prev_thresholds = np.full(num_classes, 0.5)

    thresholds = np.copy(prev_thresholds)

    for c in range(num_classes):
        best_t = prev_thresholds[c]
        best_f1 = 0.0

        # Define local search range
        t_min = max(0.0, prev_thresholds[c] - delta)
        t_max = min(1.0, prev_thresholds[c] + delta)

        # Search only within the bounded window
        for t in np.linspace(t_min, t_max, 21):  # smaller grid
            preds_c = (probs[:, c] >= t).astype(int)
            f1 = f1_score(labels[:, c], preds_c, average = "macro" , zero_division=0)

            if f1 > best_f1:
                best_f1 = f1
                best_t = t

        thresholds[c] = best_t

    return thresholds

In [21]:
training_args = TrainingArguments(
    output_dir="./results",

    learning_rate = 2e-5,

    per_device_train_batch_size = 32,
    per_device_eval_batch_size = 32,

    num_train_epochs = 20,  # increase for LoRA

    eval_strategy="epoch",
    save_strategy="epoch",

    logging_dir="./logs",
    logging_steps = 50,

    fp16= False,
    report_to="none",

    # NEW IMPORTANT PARTS
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    save_total_limit=3,  # keep only last 3 checkpoints
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [36]:
import numpy as np
from sklearn.metrics import f1_score, precision_score, recall_score

class MetricsWithDynamicThreshold:
    def __init__(self):
        self.thresholds = (0.5,) * 28  # will be updated each eval

    def __call__(self, eval_pred):
        logits, labels = eval_pred
        probs = torch.sigmoid(torch.tensor(logits)).numpy()
        labels = np.array(labels)
        
        # 🔥 Use previous thresholds to constrain search
        best_thresholds = tune_thresholds(
            probs,
            labels,
            prev_thresholds=self.thresholds,
            delta=0.1
        )

        preds = (probs >= best_thresholds).astype(int)
        self.thresholds = best_thresholds  # store latest

        precision = precision_score(labels, preds, average="macro", zero_division=0)
        recall = recall_score(labels, preds, average="macro", zero_division=0)
        micro_f1 = f1_score(labels, preds, average="micro", zero_division=0)
        macro_f1 = f1_score(labels, preds, average="macro", zero_division=0)
        weighted_f1 = f1_score(labels, preds, average="weighted", zero_division=0)

        return {
            "precision": precision,
            "recall" : recall,
            "micro_f1": micro_f1,
            "macro_f1": macro_f1,
            "weighted_f1": weighted_f1,
            "thresholds": self.thresholds.tolist(),  # important!
        }

In [37]:
metrics_obj = MetricsWithDynamicThreshold()

In [23]:
from transformers import TrainerCallback
import os
import json

class ThresholdCallback(TrainerCallback):
    def __init__(self, metrics_obj):
        self.metrics_obj = metrics_obj

    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        if metrics is None or "thresholds" not in metrics:
            return

        ckpt_dir = os.path.join(
            args.output_dir,
            f"checkpoint-{state.global_step}"
        )
        os.makedirs(ckpt_dir, exist_ok=True)

        with open(os.path.join(ckpt_dir, "thresholds.json"), "w") as f:
            json.dump(metrics["thresholds"], f)

        # keep metrics object in sync (for next eval/inference)
        self.metrics_obj.thresholds = metrics["thresholds"]

In [24]:
from sklearn.metrics import f1_score, precision_score, recall_score
def compute_metrics_no_threshold_update(eval_pred):

    logits, labels = eval_pred
    probs = torch.sigmoid(torch.tensor(logits)).numpy()
    labels = np.array(labels)

    preds = (probs >= 0.5).astype(int)

    precision = precision_score(labels, preds, average="macro", zero_division=0)
    recall = recall_score(labels, preds, average="macro", zero_division=0)
    micro_f1 = f1_score(labels, preds, average="micro", zero_division=0)
    macro_f1 = f1_score(labels, preds, average="macro", zero_division=0)
    weighted_f1 = f1_score(labels, preds, average="weighted", zero_division=0)

    return {
        "precision": precision,
        "recall" : recall,
        "micro_f1": micro_f1,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1,
    }

In [25]:
from transformers import DataCollatorWithPadding
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)


trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics_no_threshold_update,
    #callbacks=[ThresholdCallback(metrics_obj)]
)

In [26]:
trainer.train()

Epoch,Training Loss,Validation Loss,Precision,Recall,Micro F1,Macro F1,Weighted F1
1,0.389904,0.381316,0.379140,0.137999,0.298084,0.171339,0.242886
2,0.361073,0.354256,0.470190,0.235060,0.385942,0.277601,0.337630
3,0.339401,0.337707,0.503301,0.316104,0.436156,0.354871,0.398420
4,0.324997,0.325310,0.507205,0.347628,0.452814,0.383488,0.420078
5,0.315543,0.318770,0.539671,0.377389,0.474667,0.413295,0.448704
6,0.308873,0.310729,0.503976,0.406253,0.484465,0.432368,0.464557
7,0.299682,0.305649,0.492297,0.421941,0.490515,0.442724,0.474437
8,0.297133,0.301896,0.506726,0.430615,0.498321,0.454583,0.484651
9,0.290952,0.300062,0.488805,0.444012,0.498886,0.453472,0.484982
10,0.288489,0.297056,0.498961,0.452174,0.503893,0.465249,0.492951


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector

TrainOutput(global_step=5180, training_loss=0.29826266480228614, metrics={'train_runtime': 3690.1073, 'train_samples_per_second': 89.596, 'train_steps_per_second': 1.404, 'total_flos': 2.258189555920896e+16, 'train_loss': 0.29826266480228614, 'epoch': 20.0})

In [28]:
predictions_dev = trainer.predict(dataset["validation"])

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


In [30]:
def find_best_thresholds_per_label(y_true, y_prob, step=0.01):
    y_true = np.vstack(y_true)
    n_labels = y_true.shape[1]
    thresholds = np.zeros(n_labels)

    for j in range(n_labels):
        best_f1 = 0.0
        best_t = 0.5

        for t in np.arange(0.05, 0.95, step):
            y_pred = (y_prob[:, j] >= t).astype(int)

            f1 = f1_score(y_true[:, j], y_pred)

            if f1 > best_f1:
                best_f1 = f1
                best_t = t

        thresholds[j] = best_t

    return thresholds

In [31]:
import numpy as np

logits_dev = predictions_dev.predictions
probs_dev = 1 / (1 + np.exp(-logits_dev))  # sigmoid

dev_labels = dataset['validation']['labels']

thresholds = find_best_thresholds_per_label(dev_labels, probs_dev)

print(thresholds)



[0.42 0.43 0.51 0.7  0.45 0.47 0.57 0.61 0.33 0.67 0.48 0.5  0.38 0.38
 0.6  0.37 0.58 0.28 0.43 0.74 0.39 0.57 0.66 0.37 0.5  0.39 0.37 0.35]


In [36]:
predictions_test = trainer.predict(dataset["test"])

logits_test = predictions_test.predictions
probs_test = 1 / (1 + np.exp(-logits_test))  # sigmoid

preds = (probs_test > thresholds).astype(int)

labels = dataset["test"]['labels']


precision = precision_score(labels, preds, average="macro", zero_division=0)
recall = recall_score(labels, preds, average="macro", zero_division=0)
micro_f1 = f1_score(labels, preds, average="micro", zero_division=0)
macro_f1 = f1_score(labels, preds, average="macro", zero_division=0)
weighted_f1 = f1_score(labels, preds, average="weighted", zero_division=0)
print(precision,recall,micro_f1,macro_f1,weighted_f1)

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


0.48978783271523524 0.5441431950689185 0.5175020351203629 0.5043406409377503 0.5313077240140979


In [48]:
import pandas as pd
df = pd.DataFrame({"preds_labels" :list(preds)})
df.to_csv("/kaggle/working/predictions.csv", index=False)

In [51]:
print(df)

                                           preds_labels
0     [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, ...
1     [1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...
2     [0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, ...
3     [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...
4     [0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 1, 0, 0, ...
...                                                 ...
2062  [1, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, ...
2063  [0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...
2064  [0, 0, 0, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, ...
2065  [1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...
2066  [0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, ...

[2067 rows x 1 columns]


In [50]:
from IPython.display import FileLink
FileLink(r'predictions.csv')

/kaggle/working/predictions.csv

In [49]:
import os

print(os.listdir("/kaggle/working"))

['state.db', 'my_model', 'results', '.virtual_documents', 'predictions.csv', 'predictions_result_LoRA.csv', 'my_model.zip']


In [40]:
predictions = trainer.predict(dataset["test"])

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


In [44]:
import numpy as np

logits = predictions.predictions
probs = 1 / (1 + np.exp(-logits))  # sigmoid

thresholds = metrics_obj.thresholds

preds = (probs > thresholds).astype(int)

labels = dataset["test"]['labels']

precision = precision_score(labels, preds, average="macro", zero_division=0)
recall = recall_score(labels, preds, average="macro", zero_division=0)
micro_f1 = f1_score(labels, preds, average="micro", zero_division=0)
macro_f1 = f1_score(labels, preds, average="macro", zero_division=0)
weighted_f1 = f1_score(labels, preds, average="weighted", zero_division=0)

In [42]:
# Adaptive Thresholds per epoch results
# Retained here after changed trainer compute_metrics
print(precision,recall,micro_f1,macro_f1,weighted_f1)

0.5079538815459428 0.539240286247481 0.5358863198458574 0.5168794253296021 0.542332066625759


In [43]:
preds = (probs > 0.5).astype(int)
precision = precision_score(labels, preds, average="macro", zero_division=0)
recall = recall_score(labels, preds, average="macro", zero_division=0)
micro_f1 = f1_score(labels, preds, average="micro", zero_division=0)
macro_f1 = f1_score(labels, preds, average="macro", zero_division=0)
weighted_f1 = f1_score(labels, preds, average="weighted", zero_division=0)
print(precision,recall,micro_f1,macro_f1,weighted_f1)

0.25923500246612635 0.820737701191227 0.3965601965601966 0.3833209322489225 0.434849004887424


In [63]:
!cp -r /kaggle/working/results/checkpoint-5180 /kaggle/working/my_model
!zip -r my_model.zip /kaggle/working/my_model

  adding: kaggle/working/my_model/ (stored 0%)
  adding: kaggle/working/my_model/optimizer.pt (deflated 8%)
  adding: kaggle/working/my_model/adapter_model.safetensors (deflated 7%)
  adding: kaggle/working/my_model/tokenizer_config.json (deflated 48%)
  adding: kaggle/working/my_model/README.md (deflated 66%)
  adding: kaggle/working/my_model/tokenizer.json (deflated 87%)
  adding: kaggle/working/my_model/adapter_config.json (deflated 57%)
  adding: kaggle/working/my_model/rng_state.pth (deflated 26%)
  adding: kaggle/working/my_model/training_args.bin (deflated 53%)
  adding: kaggle/working/my_model/scheduler.pt (deflated 61%)
  adding: kaggle/working/my_model/trainer_state.json (deflated 81%)


In [65]:
from IPython.display import FileLink
FileLink(r'my_model.zip')


/kaggle/working/my_model.zip